# RoadSense AI: Road Damage Severity Classification & Evaluation Pipeline
**Author:** RoadSense AI Engineering Team  
**Random State:** 42 (Enforced across all splits, models, and cross-validation)  
**Dataset Foundation:** RDD2022 (Road Damage Detection Challenge 2022, Seki Lab / Figshare DOI: 10.6084/m9.figshare.21431547)

---
### Abstract & Scientific Framing
This notebook documents the machine learning pipeline for RoadSense AI. The model classifies road pavement damage into three triage severity levels: **Low**, **Medium**, and **High**. 

> **Important Scientific Note:**  
> In the absence of certified in-situ civil engineering pavement condition ratings for every photograph in the RDD2022 challenge dataset, severity labels are derived using **deterministic engineering heuristic proxy rules** based on defect counts and bounding box surface area ratios. Accuracy scores reported herein reflect **agreement with these proxy rules under 5% injected border ambiguity noise**, rather than certified ground-truth structural pavement soundness.

## 1. Dataset Description & Distribution
The Road Damage Dataset 2022 (RDD2022) comprises smartphone dashboard camera photographs captured across six countries: **Japan**, **India**, **Czech Republic**, **Norway**, **United States**, and **China**. 

Four primary road damage classes are categorized under the Pascal VOC annotation format:
- `D00`: Longitudinal Linear Cracks (aged asphalt & joint fatigue)
- `D10`: Transverse Linear Cracks (thermal contraction joints)
- `D20`: Alligator / Fatigue Cracks (interconnected base structural failure)
- `D40`: Potholes (localized cavity depressions with high tire impact hazard)

**Handling of Zero-Defect Images:** Frames with no annotated bounding boxes represent pristine or undamaged road surfaces. In the tabular feature extraction pipeline, these are mapped to zero counts across all defect categories and zero area ratio, yielding a baseline `Low` severity triage status.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from backend.ml.rdd_annotation_parser import FEATURE_COLUMNS, heuristic_severity
from backend.ml.prepare_dataset import DATA_DIR

dataset_path = os.path.join(DATA_DIR, "dataset_features.csv")
df = pd.read_csv(dataset_path)
print(f"Loaded dataset: {len(df)} samples, {len(FEATURE_COLUMNS)} structural features.")
df.head()

## 2. Exploratory Data Analysis (EDA)
Let us inspect the distribution of defect counts, area ratios, and severity target classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Severity Class Distribution
sns.countplot(x="severity", data=df, order=["Low", "Medium", "High"], palette=["#3b82f6", "#f59e0b", "#ef4444"], ax=axes[0])
axes[0].set_title("Severity Proxy Class Distribution", fontweight="bold")
axes[0].set_ylabel("Sample Count")

# 2. Defect Type Frequencies
defect_counts = df[["pothole_count", "alligator_crack_count", "longitudinal_crack_count", "transverse_crack_count"]].sum()
defect_counts.plot(kind="bar", color="#10b981", ax=axes[1])
axes[1].set_title("Total Defect Category Instances", fontweight="bold")
axes[1].set_xticklabels(["Potholes (D40)", "Alligator (D20)", "Longitudinal (D00)", "Transverse (D10)"], rotation=25, ha="right")

# 3. Largest Bounding Box Area Ratio Distribution
sns.histplot(df["largest_bbox_area_ratio"], bins=25, kde=True, color="#6366f1", ax=axes[2])
axes[2].set_title("Largest Bounding Box Area Ratio Distribution", fontweight="bold")
axes[2].set_xlabel("BBox Area / Image Area")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap across the 7 engineered features
plt.figure(figsize=(8, 6))
corr = df[FEATURE_COLUMNS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix of Extracted Damage Features", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Proxy Label Generation
The 7 structural defect features extracted from bounding boxes are:
1. `damage_count`: Total number of localized defect bounding boxes.
2. `pothole_count`: Count of deep cavity depressions (D40).
3. `alligator_crack_count`: Count of fatigue crack networks (D20).
4. `longitudinal_crack_count`: Count of longitudinal cracks (D00).
5. `transverse_crack_count`: Count of transverse cracks (D10).
6. `total_bbox_area_ratio`: Combined area of all bounding boxes divided by frame area.
7. `largest_bbox_area_ratio`: Area of the single largest defect divided by frame area.

**Train / Test Split:** An 80/20 stratified split (`random_state=42`) is utilized throughout the evaluation (960 training samples, 240 held-out test samples).

## 4. Multi-Model Benchmark, 5-Fold Cross-Validation & Hyperparameter Tuning
We evaluate 6 distinct architectures on the identical split, run 5-fold Stratified Cross-Validation on the training set, and perform hyperparameter grid search for Random Forest.

In [ ]:
# Load executed model comparison results from Task 2 report
reports_dir = os.path.join(project_root, "backend", "ml", "reports")
comp_json_path = os.path.join(reports_dir, "model_comparison.json")

if os.path.exists(comp_json_path):
    with open(comp_json_path, "r") as f:
        comp_data = json.load(f)
        
    comp_rows = []
    for m_name, d in comp_data.items():
        if m_name == "feature_importances":
            continue
        comp_rows.append({
            "Model": m_name,
            "5-Fold CV Acc": f"{d.get('cv_accuracy_mean', 0)*100:.2f}% ± {d.get('cv_accuracy_std', 0)*100:.2f}%" if 'cv_accuracy_mean' in d else f"{d.get('best_cv_macro_f1', 0)*100:.2f}% (CV F1)",
            "5-Fold CV F1": f"{d.get('cv_macro_f1_mean', 0):.4f} ± {d.get('cv_macro_f1_std', 0):.4f}" if 'cv_macro_f1_mean' in d else f"{d.get('best_cv_macro_f1', 0):.4f}",
            "Test Agreement Acc": f"{d.get('test_agreement_accuracy', 0)*100:.2f}%",
            "Test Macro-F1": f"{d.get('test_macro_f1', 0):.4f}"
        })
    display(pd.DataFrame(comp_rows))
else:
    print("Comparison JSON not found. Running benchmark directly...")
    from backend.ml.compare_models import run_model_comparison
    run_model_comparison()

In [ ]:
# Display Learning Curve, Confusion Matrices, and Feature Importance
from IPython.display import Image, display

cm_img = os.path.join(reports_dir, "confusion_matrices.png")
lc_img = os.path.join(reports_dir, "learning_curve.png")
fi_img = os.path.join(reports_dir, "feature_importance.png")
shap_img = os.path.join(reports_dir, "shap_summary.png")

if os.path.exists(cm_img):
    print("Held-Out Confusion Matrices across All Benchmark Models:")
    display(Image(cm_img))

if os.path.exists(lc_img):
    print("Learning Curve for Random Forest Severity Classifier:")
    display(Image(lc_img))

if os.path.exists(fi_img):
    print("Feature Importance: Gini Impurity vs Permutation Importance (n=20):")
    display(Image(fi_img))

if os.path.exists(shap_img):
    print("SHAP Multi-Class TreeExplainer Summary Plot:")
    display(Image(shap_img))

## 5. Feature & Feature-Group Ablation Study
To measure the true marginal contribution of individual features and grouped features (Counts vs Area Ratios), we systematically remove each and evaluate the drop in agreement accuracy and macro-F1.

In [ ]:
ablation_json_path = os.path.join(reports_dir, "ablation.json")
ablation_img_path = os.path.join(reports_dir, "ablation.png")

if os.path.exists(ablation_json_path):
    with open(ablation_json_path, "r") as f:
        ablation_data = json.load(f)
    print(f"Baseline Agreement Accuracy: {ablation_data['baseline']['agreement_accuracy']*100:.2f}%")
    print(f"Baseline Macro-F1:          {ablation_data['baseline']['macro_f1']:.4f}")
    display(pd.DataFrame(ablation_data["individual_ablations"]).T[["agreement_accuracy", "macro_f1", "accuracy_drop", "macro_f1_drop"]])
    if os.path.exists(ablation_img_path):
        display(Image(ablation_img_path))
else:
    print("Ablation data not found. Run python -m backend.ml.ablation")

## 6. Detector Validation & Hand-Labeled Evaluation Status
We inspect the validation reports from Tasks 3 and 4.

In [ ]:
det_eval_path = os.path.join(reports_dir, "detector_eval.json")
val_results_path = os.path.join(reports_dir, "validation_results.json")

if os.path.exists(det_eval_path):
    with open(det_eval_path, "r") as f:
        det_data = json.load(f)
    print("--- Computer Vision Detector Evaluation ---")
    print("Dataset Status:", det_data.get("dataset_status"))
    print("Train/Serve Mismatch:", det_data.get("train_serve_mismatch_analysis", {}).get("mismatch_implication"))

if os.path.exists(val_results_path):
    with open(val_results_path, "r") as f:
        val_data = json.load(f)
    print("\n--- Hand-Labeled Human Validation Evaluation ---")
    print("Status:", val_data.get("status"))
    print("Sample Size:", val_data.get("sample_size"))
else:
    print("\n[Notice] Hand-labeled validation set is awaiting human engineer annotations under backend/ml/validation/validation_labels.csv.")

## 7. Limitations and Future Work

A rigorous scientific appraisal of the RoadSense AI machine learning pipeline identifies several core limitations:

1. **Proxy Labels Instead of True Ground Truth:**  
   Severity classes (Low, Medium, High) are assigned using heuristic rules based on bounding box counts and area ratios. Model accuracy (~95.42%) measures faithful convergence to these rules, rather than structural soundness determined by destructive coring or falling weight deflectometers (FWD).

2. **Train / Serve Feature Mismatch:**  
   The severity model was trained on tabular features derived from ground-truth Pascal VOC annotations, whereas live user inference extracts features via a classical edge/morphological heuristic computer vision detector (`image_detector.py`). This produces domain shift on textured or wet road surfaces.

3. **Absence of Subsurface Structural Information:**  
   Surface photographs cannot assess base subgrade compaction, moisture infiltration, or pavement thickness. Deep alligator cracks frequently indicate foundation failure even when surface area appears moderate.

4. **Geographic and Environmental Bias:**  
   The RDD2022 dataset captures roads in specific countries (Japan, India, Czech, Norway) under prevailing daytime lighting conditions. Performance in unpaved rural roads, night-time capture, or snow cover requires further calibration.

5. **Future Work:**  
   - Replace classical heuristic CV with an end-to-end fine-tuned YOLOv8/YOLOv11 model trained directly on the full multi-national RDD2022 image corpus.
   - Collect verified civil engineer pavement condition ratings (e.g. PCI, Pavement Condition Index) to replace proxy heuristics with true human-certified labels.
   - Calibrate multi-factor priority weights using the Analytic Hierarchy Process (AHP) with local municipal engineering authorities.